<span style="float: left;padding: 1.3em">![logo](https://github.com/gw-odw/odw/blob/main/Tutorials/logo.png?raw=1)</span>

# Gravitational Wave Open Data Workshop

## Tutorial 1.1: Discovering open data from GW observatories

This notebook describes how to discover what data are available from the [Gravitational-Wave Open Science Center (GWOSC)](https://gwosc.org).
    
View this tutorial on [Google Colaboratory](https://colab.research.google.com/github/gw-odw/odw/blob/main/Tutorials/01_Accessing_Open_Data/Tuto_1.1_Discovering_Open_Data.ipynb) or launch [mybinder](https://mybinder.org/v2/gh/gw-odw/odw/HEAD).


## Installation (execute only if running on a cloud platform, like Google Colab, or if you haven't done the installation already!)

First, we need to install the software, which we do following the instruction in [Software Setup Instructions](../../setup.md):

> ⚠️ **Warning**: restart the runtime after running the cell below.
>
> To do so, click "Runtime" in the menu and choose "Restart and run all".
>
> You may see error messages but installation usually works.
> If you experience problems, please [report an issue](https://github.com/gw-odw/odw/issues).

In [1]:
# -- Uncomment following line if running in Google Colab
! pip install -q 'gwosc==0.8.1'

## Initialization

In [2]:
#check the version of the package gwosc you are using
import gwosc
print(gwosc.__version__)

0.8.1


The version you get should be 0.8.1. If it's not, check that you have followed all the steps in [Software Setup Instructions](../../setup.md).

## A brief presentation of GWOSC and Open Data

Open Science is the movement to make scientific research accessible to everyone and to increase scientific collaboration. Open Science includes different movements and practices such as open data, open source software and infrastructures, open access to publications and citizen science and more. See the [wikipedia page](https://en.wikipedia.org/wiki/Open_science) for more information.

Data from the LIGO-Virgo-KAGRA (LVK) collaboration are made available to the public via the [Gravitational-Wave Open Science Center (GWOSC)](https://gwosc.org), as described in the [LIGO Data Management Plan](https://dcc.ligo.org/LIGO-M1000066/public).

For a more detailed presentation of the data, including conventions on file and channel names and details about the preparation of the data, see the paper "Open Data from LIGO, Virgo, and KAGRA through the First Part of the Fourth Observing Run" ([link](https://arxiv.org/abs/2508.18079)).

The GWOSC also provides links to software used to analyze LVK data and organize training sessions (you are participating in it!).

## Querying for event information

The module `gwosc.datasets` provides tools for searching for datasets, including events, catalogs and full run strain data releases.


For example, we can search for events in the [GWTC-1 catalog](https://gwosc.org/eventapi/html/GWTC-1-confident/), the catalog of all events from the O1 and O2 observing runs. A list of available catalogs can be seen in the [Event Portal](https://gwosc.org/eventapi)

In [3]:
from gwosc.datasets import find_datasets
from gwosc import datasets

#-- List all available catalogs
print("List of available catalogs")
print(find_datasets(type="catalog"))

List of available catalogs
['GWTC', 'GWTC-1-confident', 'GWTC-1-marginal', 'GWTC-2', 'GWTC-2.1-auxiliary', 'GWTC-2.1-confident', 'GWTC-2.1-marginal', 'GWTC-3-confident', 'GWTC-3-marginal', 'GWTC-4.0', 'IAS-O3a', 'Initial_LIGO_Virgo', 'O1_O2-Preliminary', 'O3_Discovery_Papers', 'O3_IMBH_marginal', 'O4_Discovery_Papers']


In [4]:
#-- Print all the GW events from the GWTC-1 catalog
gwtc1 = datasets.find_datasets(type='events', catalog='GWTC-1-confident')
print('GWTC-1 events:', gwtc1)
print("")

GWTC-1 events: ['GW150914-v3', 'GW151012-v3', 'GW151226-v2', 'GW170104-v2', 'GW170608-v3', 'GW170729-v1', 'GW170809-v1', 'GW170814-v3', 'GW170817-v3', 'GW170818-v1', 'GW170823-v1']



Note that the event name is of the type _GWyymmdd-vx_ where _x_ is the last available version for the data set provided by GWOSC.

In [5]:
#-- Print all the large strain data sets from LIGO/Virgo/KAGRA observing runs
runs = find_datasets(type='run')
print('Large data sets:', runs)

Large data sets: ['O1', 'O2', 'O3GK', 'O3a', 'O3b', 'O4a', 'O4b1Disc', 'O4b2Disc', 'O4b3Disc', 'S5', 'S6']


_Attention: Observing runs whose names end with "Disc" are not real runs but only structures we add in our database for technical reasons and are associated with GW candidate detections published before the data release of the associated observing run. The papers about these detections are called in our jargon "discovery papers", hence the "Disc" suffix (e.g. O4b1Disc -> GW241011)_

`datasets.find_datasets` also accepts a `segment` and `detector` keyword to narrow results based on GPS time and detector:

In [6]:
#-- Detector and segments keywords limit search result
print(datasets.find_datasets(type='events', catalog='GWTC-1-confident', detector="L1", segment=(1164556817, 1187733618)))

['GW170104-v2', 'GW170608-v3', 'GW170729-v1', 'GW170809-v1', 'GW170814-v3', 'GW170817-v3', 'GW170818-v1', 'GW170823-v1']


Using `gwosc.datasets.event_gps`, we can query for the GPS time of a specific event (it works also without the version number):

In [7]:
from gwosc.datasets import event_gps
gps = event_gps('GW190425')
print(gps)

1240215503.0


<div class="alert alert-info">All of these times are returned in the GPS time system, which counts the number of seconds that have elapsed since the start of the GPS epoch at midnight (00:00) on January 6th 1980. GWOSC provides a <a href="https://gwosc.org/gps/">GPS time converter</a> you can use to translate into datetime, or you can use <a href="https://gwpy.readthedocs.io/en/v3.0.14/time/"><code>gwpy.time</code></a>.</div>

In [ ]:
# You can do also the vice-versa
from gwosc.datasets import event_at_gps
print(datasets.event_at_gps(1240215503))

GW190425


Note that the method `event_at_gps` looks for events found within 1 seconds of the given GPS time. If no events is found it will give an error.

We can query for the GPS time interval for an observing run:

In [8]:
from gwosc.datasets import run_segment
print(run_segment('O1'))

(1126051217, 1137254417)


In [9]:
# and vice-versa also in this case
from gwosc.datasets import run_at_gps
print(run_at_gps(1240215503))

O3a


Now we can use what we have learned with `run_segment` and `find_datasets` to see only the confident events in O1:

In [10]:
O1_events = datasets.find_datasets(type='events', catalog='GWTC-1-confident', segment=run_segment('O1'))
print(O1_events)

['GW150914-v3', 'GW151012-v3', 'GW151226-v2']


## Querying for data files

The `gwosc.locate` module provides a function to find the URLs of data files associated with a given dataset.

For event datasets, one can get the list of URLs using only the event name:

In [11]:
from gwosc.locate import get_event_urls
urls = get_event_urls('GW150914')
print(urls)

['https://gwosc.org/archive/data/O1/1126170624/H-H1_LOSC_4_V1-1126256640-4096.hdf5', 'https://gwosc.org/archive/data/O1/1126170624/L-L1_LOSC_4_V1-1126256640-4096.hdf5']


By default, this function returns all of the files associated with a given event, which isn't particularly helpful. However, we can filter on any of these by using keyword arguments, for example to get the URL for the 32-second file for the LIGO-Livingston detector:

In [ ]:
urls = get_event_urls('GW150914', duration=32, detector='L1')
print(urls)

['https://gwosc.org/archive/data/O1/1126170624/L-L1_LOSC_4_V1-1126256640-4096.hdf5']


The different interferometers are denoted by a short identifier:

- `'G1'` - GEO600
- `'H1'` - LIGO-Hanford
- `'L1'` - LIGO-Livingston
- `'V1'` - (Advanced) Virgo
- `'K1'` - KAGRA

##  Query filtered by merger parameters
The `query_events` module of `gwosc.datasets` allows to get a list of events filtered by some parameters, similar to what is done by the `Query` function of the [event portal](https://gwosc.org/eventapi/html/query/). A list of available parameters can be found [here](https://gwosc.readthedocs.io/en/stable/reference/gwosc.datasets.query_events.html) or using `query_events?`.

Let's see how to use this module to find which events have been detected with a network signal to noise ratio (SNR) between 25 and 30:

In [12]:
from gwosc.datasets import query_events
selection = query_events(select=["25 <= network-matched-filter-snr <= 30"])
#this is equivalent to
#query_events(select=["network-matched-filter-snr <= 30", "network-matched-filter-snr>= 25"])
print(selection)

['GW240105_151143-v1', 'GW230627_015337-v1', 'GW200129_065458-v1', 'GW190814-v1', 'GW190814_211039-v3', 'GW190521_074359-v2', 'GW150914-v3', 'GW150914-v4']


Note that this module will give the list of **all available versions** for all the events that have the required parameters. For example, in this query the event GW190814 is listed twice because 2 versions of that event satisfy the request of SNR between 25 and 30.

## Quiz Questions

Now that you've seen examples of how to query for dataset information using the `gwosc` package, please try and complete the following exercises using that interface and enter your response on Thinkific.

### Question 1 (no code needed, reply on Thinkific)

What can be accessed from the Gravitational Wave Open Science Center?

Strain data files, data quality segments, and event catalogs and parameter estimation of merger events.

### Question 2

How many months did O2 last?

In [29]:
from gwosc.datasets import run_segment
start, stop = run_segment('O2') #time of the beginning and end of the run

run_time = (stop - start)/(3600*24*30) #months
print(f'O2 lasted {run_time:0.0f} months')

O2 lasted 9 months


### Question 3

How many GWTC-3-confident events were detected during O3b?

In [30]:
from gwosc.datasets import find_datasets
from gwosc import datasets

#list all available catalogs
print("List of available catalogs")
print(find_datasets(type="catalog"))

List of available catalogs
['GWTC', 'GWTC-1-confident', 'GWTC-1-marginal', 'GWTC-2', 'GWTC-2.1-auxiliary', 'GWTC-2.1-confident', 'GWTC-2.1-marginal', 'GWTC-3-confident', 'GWTC-3-marginal', 'GWTC-4.0', 'IAS-O3a', 'Initial_LIGO_Virgo', 'O1_O2-Preliminary', 'O3_Discovery_Papers', 'O3_IMBH_marginal', 'O4_Discovery_Papers']


In [35]:
#Print all the GW events from the GWTC-3-confident during O3b
O3b_events = datasets.find_datasets(type='events', catalog='GWTC-3-confident', segment=run_segment('O3b'))
number_O3b_events = len(O3b_events)

print(f'03b confident events:  {number_O3b_events}')

03b confident events:  35


### Question 4

How many events have been detected with a network signal to noise ratio (SNR) >= 30? Hint: pay attention to the events' name.

In [37]:
from gwosc.datasets import query_events

selection = query_events(select=["network-matched-filter-snr>= 30"])
print(selection)

['GW231226_101520-v1', 'GW230814_230901-v1', 'GW230814_230901-v2', 'GW170817-v3']


3 events have been detected with a SNR >= 30